# Capitolo 14 — Ricorsione (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_python.ipynb)

Questo notebook accompagna il capitolo 14 «Ricorsione» delle dispense (volume 4, *Analisi e ricorsione*) e della [pagina del sito](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/). Contiene il codice **Python** degli algoritmi con pseudocodice del capitolo: `Factorial`, `Fib`, `IterativeFib`, `RecursiveGCD`, `FastPower`, `Hanoi`. Per ogni algoritmo ricorsivo il codice conta le **chiamate** (i nodi dell'albero delle chiamate), la **profondità** della ricorsione (l'altezza massima dello stack) e i **passi** (esecuzioni delle righe, numerate come nelle dispense: la riga 1 è l'intestazione `function`), e stampa la **traccia** dell'esecuzione: le chiamate in ordine di visita in profondità, con il rientro che indica la posizione nello stack. Gli esempi sono quelli del capitolo.

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime ▸ Esegui tutto* (oppure cella per cella con Maiusc+Invio). Non serve installare nulla.

Ogni algoritmo è una funzione Python; le celle di esecuzione lo lanciano sugli stessi esempi delle dispense e stampano i risultati.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno i nomi dello pseudocodice. Due eccezioni in Python, dove `from` e `next` hanno un significato speciale: `Hanoi` usa `frm` al posto di `from` e `IterativeFib` usa `nxt` al posto di `next`.

## Strumenti: contatori, stack e traccia

La cella definisce i contatori (`cnt`, `calls`, `max_depth`), lo stack delle chiamate (`push` e `pop`) e le funzioni che stampano la traccia (`event`, `say`) e la tabella dei passi (`print_steps`). Basta eseguirla una volta.

In [1]:
cnt = [0] * 20      # cnt[r] = numero di esecuzioni della riga r (righe numerate come nelle dispense)
calls = 0           # numero di chiamate = nodi dell'albero delle chiamate
max_depth = 0       # profondita' della ricorsione = altezza massima dello stack
mults = 0           # moltiplicazioni (FastPower)
moves = 0           # mosse (Hanoi)
ev = 0              # contatore degli eventi della traccia
stack = []          # lo stack delle chiamate: gli argomenti delle chiamate attive
seen = [0] * 64     # seen[x] = numero di chiamate con argomento x
trace = False       # True: stampa la traccia dell'esecuzione


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # inizia una chiamata: il suo frame va in cima allo stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # la chiamata termina: il suo frame esce dallo stack
    stack.pop()


def event(text):                   # riga numerata della traccia, con lo stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Riga | Esecuzioni")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Passi totali S = %d" % sum(cnt))

## 1. Fattoriale: Factorial

**Input:** un numero naturale $n$. **Output:** $n!$, con $0! = 1$. $S(n) = 2n + 2 = \Theta(n)$ passi; $n + 1$ chiamate, profondità $n + 1$ (l'albero è un cammino).

Pseudocodice: sezione 14.4 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
def factorial(n):
    push(n); cnt[2] += 1
    if n == 0:
        cnt[3] += 1
        if trace: event("Factorial(0): n = 0 => caso base")
        pop()
        if trace: event("Factorial(0) restituisce 1")
        return 1
    if trace: event("Factorial(%d): %d != 0 => deve calcolare %d * Factorial(%d) e attende" % (n, n, n, n - 1))
    cnt[5] += 1
    f = factorial(n - 1)
    r = n * f
    pop()
    if trace: event("Factorial(%d) restituisce %d * %d = %d" % (n, n, f, r))
    return r

In [3]:
# Esempio delle dispense: Factorial(4), con lo stack
n = 4
reset()
trace = True
r = factorial(n)
trace = False
print("Factorial(%d) = %d" % (n, r))
print("Chiamate = %d (nodi dell'albero), profondita' della ricorsione = %d" % (calls, max_depth))
print_steps(2, 3, 5)
print("S(n) = 2n + 2 = %d" % (2 * n + 2))

# Chiamate, profondita' e passi al variare di n
print()
print("  n |                   n! | chiamate | profondita' | S(n) | 2n + 2")
for n in [0, 1, 2, 3, 4, 5, 10, 15, 20]:
    reset()
    f = factorial(n)
    print("%3d | %20d | %8d | %11d | %4d | %d" % (n, f, calls, max_depth, sum(cnt), 2 * n + 2))

 1. Factorial(4): 4 != 0 => deve calcolare 4 * Factorial(3) e attende   stack: [4]
 2. Factorial(3): 3 != 0 => deve calcolare 3 * Factorial(2) e attende   stack: [4, 3]
 3. Factorial(2): 2 != 0 => deve calcolare 2 * Factorial(1) e attende   stack: [4, 3, 2]
 4. Factorial(1): 1 != 0 => deve calcolare 1 * Factorial(0) e attende   stack: [4, 3, 2, 1]
 5. Factorial(0): n = 0 => caso base   stack: [4, 3, 2, 1, 0]
 6. Factorial(0) restituisce 1   stack: [4, 3, 2, 1]
 7. Factorial(1) restituisce 1 * 1 = 1   stack: [4, 3, 2]
 8. Factorial(2) restituisce 2 * 1 = 2   stack: [4, 3]
 9. Factorial(3) restituisce 3 * 2 = 6   stack: [4]
10. Factorial(4) restituisce 4 * 6 = 24   stack: []
Factorial(4) = 24
Chiamate = 5 (nodi dell'albero), profondita' della ricorsione = 5
Riga | Esecuzioni
   2 |          5
   3 |          1
   5 |          4
Passi totali S = 10
S(n) = 2n + 2 = 10

  n |                   n! | chiamate | profondita' | S(n) | 2n + 2
  0 |                    1 |        1 |           1 | 

**Prova tu.** Cambia $n$ e guarda lo stack crescere e poi svuotarsi.

In [4]:
n = 6        # <-- cambia n (al massimo 20)
reset()
trace = True
r = factorial(n)
trace = False
print("Factorial(%d) = %d, chiamate = %d, profondita' = %d, S = %d" % (n, r, calls, max_depth, sum(cnt)))

 1. Factorial(6): 6 != 0 => deve calcolare 6 * Factorial(5) e attende   stack: [6]
 2. Factorial(5): 5 != 0 => deve calcolare 5 * Factorial(4) e attende   stack: [6, 5]
 3. Factorial(4): 4 != 0 => deve calcolare 4 * Factorial(3) e attende   stack: [6, 5, 4]
 4. Factorial(3): 3 != 0 => deve calcolare 3 * Factorial(2) e attende   stack: [6, 5, 4, 3]
 5. Factorial(2): 2 != 0 => deve calcolare 2 * Factorial(1) e attende   stack: [6, 5, 4, 3, 2]
 6. Factorial(1): 1 != 0 => deve calcolare 1 * Factorial(0) e attende   stack: [6, 5, 4, 3, 2, 1]
 7. Factorial(0): n = 0 => caso base   stack: [6, 5, 4, 3, 2, 1, 0]
 8. Factorial(0) restituisce 1   stack: [6, 5, 4, 3, 2, 1]
 9. Factorial(1) restituisce 1 * 1 = 1   stack: [6, 5, 4, 3, 2]
10. Factorial(2) restituisce 2 * 1 = 2   stack: [6, 5, 4, 3]
11. Factorial(3) restituisce 3 * 2 = 6   stack: [6, 5, 4]
12. Factorial(4) restituisce 4 * 6 = 24   stack: [6, 5]
13. Factorial(5) restituisce 5 * 24 = 120   stack: [6]
14. Factorial(6) restituisce 6 * 120

## 2. Numeri di Fibonacci: Fib

**Input:** un numero naturale $n$. **Output:** $F(n)$, con $F(0) = 0$, $F(1) = 1$ e $F(n) = F(n-1) + F(n-2)$. Algoritmo ricorsivo naive: $C(n) = 2F(n+1) - 1$ chiamate e $S(n) = 4F(n+1) - 2 = 2^{\Theta(n)}$ passi.

Pseudocodice: sezione 14.5 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [5]:
def fib(n):
    push(n); cnt[2] += 1
    if n <= 1:
        cnt[3] += 1
        if trace: say("Fib(%d)" % n, "caso base, restituisce %d" % n)
        pop()
        return n
    if trace: say("Fib(%d)" % n, "chiama Fib(%d)" % (n - 1))
    x = fib(n - 1)                                 # le due chiamate sono separate solo per stampare la traccia
    if trace: say("Fib(%d)" % n, "chiama Fib(%d)" % (n - 2))
    y = fib(n - 2)
    cnt[5] += 1
    if trace: say("Fib(%d)" % n, "restituisce %d + %d = %d" % (x, y, x + y))
    pop()
    return x + y

In [6]:
def fib_value(k):                                  # F(k) con un ciclo, per controllare
    a, b = 0, 1
    for _ in range(k):
        a, b = b, a + b
    return a


# Esempio delle dispense: l'esecuzione di Fib(4), in ordine di visita in profondita'
n = 4
reset()
trace = True
print("%-28s| %s" % ("chiamata", "azione"))
r = fib(n)
trace = False
print("F(%d) = %d, chiamate C(%d) = %d = 2F(%d) - 1, profondita' = %d" % (n, r, n, calls, n + 1, max_depth))

# Fib(5): gli stessi sottoproblemi sono risolti piu' volte
print()
n = 5
reset()
r = fib(n)
print("Fib(%d) = %d: %d chiamate, profondita' %d, S = %d passi" % (n, r, calls, max_depth, sum(cnt)))
print("  k | chiamate di Fib(k)")
for k in range(n, -1, -1):
    print("%3d | %d" % (k, seen[k]))
print_steps(2, 3, 5)

# Numero di chiamate e di passi: C(n) = 2F(n+1) - 1 e S(n) = 4F(n+1) - 2
print()
print("  n |   F(n) |     C(n) | 2F(n+1) - 1 |     S(n) | 4F(n+1) - 2 | profondita'")
for n in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30]:
    reset()
    r = fib(n)
    print("%3d | %6d | %8d | %11d | %8d | %11d | %d" % (n, r, calls, 2 * fib_value(n + 1) - 1, sum(cnt), 4 * fib_value(n + 1) - 2, max_depth))

chiamata                    | azione
Fib(4)                      | chiama Fib(3)
  Fib(3)                    | chiama Fib(2)
    Fib(2)                  | chiama Fib(1)
      Fib(1)                | caso base, restituisce 1
    Fib(2)                  | chiama Fib(0)
      Fib(0)                | caso base, restituisce 0
    Fib(2)                  | restituisce 1 + 0 = 1
  Fib(3)                    | chiama Fib(1)
    Fib(1)                  | caso base, restituisce 1
  Fib(3)                    | restituisce 1 + 1 = 2
Fib(4)                      | chiama Fib(2)
  Fib(2)                    | chiama Fib(1)
    Fib(1)                  | caso base, restituisce 1
  Fib(2)                    | chiama Fib(0)
    Fib(0)                  | caso base, restituisce 0
  Fib(2)                    | restituisce 1 + 0 = 1
Fib(4)                      | restituisce 2 + 1 = 3
F(4) = 3, chiamate C(4) = 9 = 2F(5) - 1, profondita' = 4

Fib(5) = 5: 15 chiamate, profondita' 5, S = 30 passi
  k | chiamate di

**Prova tu.** Cambia $n$ e guarda quanto crescono le chiamate.

In [7]:
n = 6        # <-- cambia n (con n > 35 ci vuole un po')
reset()
r = fib(n)
print("F(%d) = %d, chiamate = %d, S = %d" % (n, r, calls, sum(cnt)))

F(6) = 8, chiamate = 25, S = 50


## 3. Fibonacci iterativo: IterativeFib

**Input:** un numero naturale $n$. **Output:** $F(n)$, calcolato da $F(0)$ e $F(1)$ in su tenendo solo gli ultimi due valori. $S(n) = 5n + 1 = \Theta(n)$ passi. Invariante: prima di ogni valutazione della condizione, $prev = F(i-2)$ e $curr = F(i-1)$.

Pseudocodice: sezione 14.5.2 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [8]:
def iterative_fib(n, trace=False):
    cnt[2] += 1
    if n <= 1:
        cnt[3] += 1
        return n
    prev = 0; cnt[5] += 1
    curr = 1; cnt[6] += 1
    i = 2; cnt[7] += 1
    if trace:                                      # traccia: riga di inizializzazione
        print("%-10s | %-8s | %-18s | %4d | %4d" % ("inizio", "2", "-", prev, curr))
    cnt[8] += 1
    while i <= n:
        nxt = prev + curr; cnt[9] += 1             # next nello pseudocodice (in Python e' una funzione predefinita)
        prev = curr; cnt[10] += 1
        curr = nxt; cnt[11] += 1
        i = i + 1; cnt[12] += 1
        cnt[8] += 1
        if trace:                                  # traccia: i - 2 e' il numero dell'iterazione
            print("%-10d | %-8s | %-18s | %4d | %4d" % (i - 2, "%d -> %d" % (i - 1, i), "%d + %d = %d" % (nxt - prev, prev, nxt), prev, curr))
    cnt[14] += 1
    return curr

In [9]:
# Esempio delle dispense: IterativeFib(4)
n = 4
reset()
print("iterazione | i        | next = prev + curr | prev | curr")
r = iterative_fib(n, True)
print("Il ciclo termina con i = %d > %d: restituisce curr = %d" % (n + 1, n, r))
print_steps(2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14)
print("S(n) = 5n + 1 = %d" % (5 * n + 1))

# Confronto con Fib: passi di Fib = 4F(n+1) - 2 (formula verificata nella sezione precedente)
print()
print("  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1")
for n in [5, 10, 20, 30]:
    reset()
    r = iterative_fib(n)
    steps = sum(cnt)
    print("%3d | %6d | %16d | %12d | %d" % (n, r, 4 * iterative_fib(n + 1) - 2, steps, 5 * n + 1))

iterazione | i        | next = prev + curr | prev | curr
inizio     | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
Il ciclo termina con i = 5 > 4: restituisce curr = 3
Riga | Esecuzioni
   2 |          1
   3 |          0
   5 |          1
   6 |          1
   7 |          1
   8 |          4
   9 |          3
  10 |          3
  11 |          3
  12 |          3
  14 |          1
Passi totali S = 21
S(n) = 5n + 1 = 21

  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1
  5 |      5 |               30 |           26 | 26
 10 |     55 |              354 |           51 | 51
 20 |   6765 |            43782 |          101 | 101
 30 | 832040 |          5385074 |          151 | 151


**Prova tu.** Cambia $n$ e controlla che i passi siano $5n + 1$.

In [10]:
n = 10        # <-- cambia n
reset()
print("iterazione | i        | next = prev + curr | prev | curr")
r = iterative_fib(n, True)
print("F(%d) = %d, S = %d = 5n + 1 = %d" % (n, r, sum(cnt), 5 * n + 1))

iterazione | i        | next = prev + curr | prev | curr
inizio     | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
4          | 5 -> 6   | 2 + 3 = 5          |    3 |    5
5          | 6 -> 7   | 3 + 5 = 8          |    5 |    8
6          | 7 -> 8   | 5 + 8 = 13         |    8 |   13
7          | 8 -> 9   | 8 + 13 = 21        |   13 |   21
8          | 9 -> 10  | 13 + 21 = 34       |   21 |   34
9          | 10 -> 11 | 21 + 34 = 55       |   34 |   55
F(10) = 55, S = 51 = 5n + 1 = 51


## 4. Massimo comun divisore: RecursiveGCD

**Input:** due numeri naturali $n \ge 1$ e $m \ge 0$. **Output:** $\gcd(n, m)$. Ricorsione di coda: ogni chiamata ne fa al più una, l'albero è un cammino. Al più $2\lfloor\log_2 m\rfloor + 3$ chiamate, $O(\log_2 m)$ passi.

Pseudocodice: sezione 14.6 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [11]:
def floor_log2(m):                                 # floor(log2 m) con un ciclo (solo per il limite sulle chiamate)
    k = 0
    while m > 1:
        m = m // 2
        k = k + 1
    return k


def recursive_gcd(n, m):
    push(m); cnt[2] += 1
    if m == 0:
        cnt[3] += 1
        if trace: say("RecursiveGCD(%d, %d)" % (n, m), "caso base, restituisce %d" % n)
        pop()
        return n
    if trace: say("RecursiveGCD(%d, %d)" % (n, m), "%d mod %d = %d: chiama RecursiveGCD(%d, %d)" % (n, m, n % m, m, n % m))
    cnt[5] += 1
    g = recursive_gcd(m, n % m)
    if trace: say("RecursiveGCD(%d, %d)" % (n, m), "restituisce %d" % g)
    pop()
    return g

In [12]:
# Esempi delle dispense: RecursiveGCD(48, 18) e RecursiveGCD(252, 105), alberi delle chiamate (cammini)
for n, m in [(48, 18), (252, 105)]:
    reset()
    trace = True
    print("%-28s| %s" % ("chiamata", "azione"))
    g = recursive_gcd(n, m)
    trace = False
    print("gcd(%d, %d) = %d, chiamate = %d, S = %d" % (n, m, g, calls, sum(cnt)))
    print()

# Numero di chiamate (ogni chiamata esegue 2 righe, quindi S = 2 chiamate) e limite 2 floor(log2 m) + 3
print("      n |       m |  gcd | chiamate |   S | 2 floor(log2 m) + 3")
for n, m in [(48, 18), (252, 105), (89, 55), (144, 89), (1000, 7), (1000000, 999999)]:
    reset()
    g = recursive_gcd(n, m)
    print("%7d | %7d | %4d | %8d | %3d | %d" % (n, m, g, calls, sum(cnt), 2 * floor_log2(m) + 3))

chiamata                    | azione
RecursiveGCD(48, 18)        | 48 mod 18 = 12: chiama RecursiveGCD(18, 12)
  RecursiveGCD(18, 12)      | 18 mod 12 = 6: chiama RecursiveGCD(12, 6)
    RecursiveGCD(12, 6)     | 12 mod 6 = 0: chiama RecursiveGCD(6, 0)
      RecursiveGCD(6, 0)    | caso base, restituisce 6
    RecursiveGCD(12, 6)     | restituisce 6
  RecursiveGCD(18, 12)      | restituisce 6
RecursiveGCD(48, 18)        | restituisce 6
gcd(48, 18) = 6, chiamate = 4, S = 8

chiamata                    | azione
RecursiveGCD(252, 105)      | 252 mod 105 = 42: chiama RecursiveGCD(105, 42)
  RecursiveGCD(105, 42)     | 105 mod 42 = 21: chiama RecursiveGCD(42, 21)
    RecursiveGCD(42, 21)    | 42 mod 21 = 0: chiama RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | caso base, restituisce 21
    RecursiveGCD(42, 21)    | restituisce 21
  RecursiveGCD(105, 42)     | restituisce 21
RecursiveGCD(252, 105)      | restituisce 21
gcd(252, 105) = 21, chiamate = 4, S = 8

      n |       m |  gcd | ch

**Prova tu.** Cambia i due numeri; il caso peggiore sono due numeri di Fibonacci consecutivi.

In [13]:
n, m = 1071, 462        # <-- cambia i due numeri
reset()
trace = True
print("%-28s| %s" % ("chiamata", "azione"))
g = recursive_gcd(n, m)
trace = False
print("gcd(%d, %d) = %d, chiamate = %d" % (n, m, g, calls))

chiamata                    | azione
RecursiveGCD(1071, 462)     | 1071 mod 462 = 147: chiama RecursiveGCD(462, 147)
  RecursiveGCD(462, 147)    | 462 mod 147 = 21: chiama RecursiveGCD(147, 21)
    RecursiveGCD(147, 21)   | 147 mod 21 = 0: chiama RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | caso base, restituisce 21
    RecursiveGCD(147, 21)   | restituisce 21
  RecursiveGCD(462, 147)    | restituisce 21
RecursiveGCD(1071, 462)     | restituisce 21
gcd(1071, 462) = 21, chiamate = 4


## 5. Elevamento a potenza veloce: FastPower

**Input:** un numero $x$ e un numero naturale $n$. **Output:** $x^n$, per quadrati ripetuti. $S(n) = 4\lfloor\log_2 n\rfloor + 6 = \Theta(\log_2 n)$ passi, $\lfloor\log_2 n\rfloor + 2$ chiamate, al più $2(\lfloor\log_2 n\rfloor + 1)$ moltiplicazioni.

Pseudocodice: sezione 14.7 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [14]:
def floor_log2(n):                                 # floor(log2 n) con un ciclo (solo per la formula dei passi)
    k = 0
    while n > 1:
        n = n // 2
        k = k + 1
    return k


def fast_power(x, n):
    global mults
    push(n); cnt[2] += 1
    if n == 0:
        cnt[3] += 1
        if trace: say("FastPower(%d, 0)" % x, "caso base, restituisce 1")
        pop()
        return 1
    if trace: say("FastPower(%d, %d)" % (x, n), "%d != 0 => chiama FastPower(%d, %d)   (%d/2 = %d)" % (n, x, n // 2, n, n // 2))
    y = fast_power(x, n // 2); cnt[5] += 1         # una sola chiamata ricorsiva: il risultato e' in y
    cnt[6] += 1
    if n % 2 == 0:
        cnt[7] += 1
        mults += 1
        r = y * y
        if trace: say("FastPower(%d, %d)" % (x, n), "y = %d, %d e' pari => restituisce %d * %d = %d" % (y, n, y, y, r))
    else:
        cnt[9] += 1
        mults += 2
        r = x * y * y
        if trace: say("FastPower(%d, %d)" % (x, n), "y = %d, %d e' dispari => restituisce %d * %d * %d = %d" % (y, n, x, y, y, r))
    pop()
    return r

In [15]:
# Esempio delle dispense: FastPower(3, 13)
x, n = 3, 13
reset()
trace = True
print("%-28s| %s" % ("chiamata", "azione"))
r = fast_power(x, n)
trace = False
print("%d^%d = %d, chiamate = %d, moltiplicazioni = %d" % (x, n, r, calls, mults))
print_steps(2, 3, 5, 6, 7, 9)
print("S(n) = 4 floor(log2 n) + 6 = %d" % (4 * floor_log2(n) + 6))

# Quante moltiplicazioni: circa 2 log2 n invece di n (x = 1, per non traboccare)
print()
print("      n | chiamate | moltiplicazioni | S(n) | 4 floor(log2 n) + 6")
for n in [1, 2, 13, 100, 1000, 1000000]:
    reset()
    fast_power(1, n)
    print("%7d | %8d | %15d | %4d | %d" % (n, calls, mults, sum(cnt), 4 * floor_log2(n) + 6))

chiamata                    | azione
FastPower(3, 13)            | 13 != 0 => chiama FastPower(3, 6)   (13/2 = 6)
  FastPower(3, 6)           | 6 != 0 => chiama FastPower(3, 3)   (6/2 = 3)
    FastPower(3, 3)         | 3 != 0 => chiama FastPower(3, 1)   (3/2 = 1)
      FastPower(3, 1)       | 1 != 0 => chiama FastPower(3, 0)   (1/2 = 0)
        FastPower(3, 0)     | caso base, restituisce 1
      FastPower(3, 1)       | y = 1, 1 e' dispari => restituisce 3 * 1 * 1 = 3
    FastPower(3, 3)         | y = 3, 3 e' dispari => restituisce 3 * 3 * 3 = 27
  FastPower(3, 6)           | y = 27, 6 e' pari => restituisce 27 * 27 = 729
FastPower(3, 13)            | y = 729, 13 e' dispari => restituisce 3 * 729 * 729 = 1594323
3^13 = 1594323, chiamate = 5, moltiplicazioni = 7
Riga | Esecuzioni
   2 |          5
   3 |          1
   5 |          4
   6 |          4
   7 |          1
   9 |          3
Passi totali S = 18
S(n) = 4 floor(log2 n) + 6 = 18

      n | chiamate | moltiplicazioni | S(n) | 4 f

**Prova tu.** Cambia $x$ e $n$ e conta le moltiplicazioni.

In [16]:
x, n = 2, 10        # <-- cambia x e n (con x e n grandi C++ e Java traboccano)
reset()
trace = True
print("%-28s| %s" % ("chiamata", "azione"))
r = fast_power(x, n)
trace = False
print("%d^%d = %d, moltiplicazioni = %d" % (x, n, r, mults))

chiamata                    | azione
FastPower(2, 10)            | 10 != 0 => chiama FastPower(2, 5)   (10/2 = 5)
  FastPower(2, 5)           | 5 != 0 => chiama FastPower(2, 2)   (5/2 = 2)
    FastPower(2, 2)         | 2 != 0 => chiama FastPower(2, 1)   (2/2 = 1)
      FastPower(2, 1)       | 1 != 0 => chiama FastPower(2, 0)   (1/2 = 0)
        FastPower(2, 0)     | caso base, restituisce 1
      FastPower(2, 1)       | y = 1, 1 e' dispari => restituisce 2 * 1 * 1 = 2
    FastPower(2, 2)         | y = 2, 2 e' pari => restituisce 2 * 2 = 4
  FastPower(2, 5)           | y = 4, 5 e' dispari => restituisce 2 * 4 * 4 = 32
FastPower(2, 10)            | y = 32, 10 e' pari => restituisce 32 * 32 = 1024
2^10 = 1024, moltiplicazioni = 6


## 6. Torri di Hanoi: Hanoi

**Input:** il numero di dischi $n \ge 1$, il piolo di partenza $from$, quello di arrivo $to$ e quello ausiliario $aux$. **Output:** la sequenza delle mosse che porta i dischi $1, \dots, n$ da $from$ a $to$. $M(n) = 2^n - 1$ mosse (ottime), $S(n) = 3 \cdot 2^n - 4 = \Theta(2^n)$ passi, profondità $n$.

Pseudocodice: sezione 14.8 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [17]:
def move(disk, frm, to):                           # "move disk d from from to to": conta la mossa e, se show_moves, la stampa
    global moves
    moves += 1
    if show_moves:
        print("%d. sposta il disco %d da %s a %s" % (moves, disk, frm, to))


show_moves = False


def hanoi(n, frm, to, aux):                        # in Python "from" e' una parola riservata: frm
    push(n); cnt[2] += 1
    call = "Hanoi(%d,%s,%s,%s)" % (n, frm, to, aux) if trace else ""
    if n == 1:
        cnt[3] += 1
        move(1, frm, to)
        if trace: say(call, "stampa: sposta il disco 1 da %s a %s, ritorna" % (frm, to))
    else:
        if trace: say(call, "chiama Hanoi(%d,%s,%s,%s)" % (n - 1, frm, aux, to))
        cnt[5] += 1
        hanoi(n - 1, frm, aux, to)
        cnt[6] += 1
        move(n, frm, to)
        if trace: say(call, "stampa: sposta il disco %d da %s a %s, chiama Hanoi(%d,%s,%s,%s)" % (n, frm, to, n - 1, aux, to, frm))
        cnt[7] += 1
        hanoi(n - 1, aux, to, frm)
        if trace: say(call, "ritorna")
    pop()

In [18]:
# Esempio delle dispense: Hanoi(3, A, C, B), le mosse
reset()
show_moves = True
hanoi(3, "A", "C", "B")
show_moves = False
print("%d mosse, %d chiamate, profondita' della ricorsione = %d" % (moves, calls, max_depth))

# La stessa esecuzione, chiamata per chiamata (rientro = profondita' nello stack)
print()
reset()
trace = True
print("%-28s| %s" % ("chiamata", "azione"))
hanoi(3, "A", "C", "B")
trace = False
print_steps(2, 3, 5, 6, 7)
print("S(n) = 3 * 2^n - 4 = %d" % (3 * 2 ** 3 - 4))

# Mosse e passi: M(n) = 2^n - 1 e S(n) = 3 * 2^n - 4
print()
print("  n |  mosse | 2^n - 1 | chiamate |   S(n) | 3 * 2^n - 4 | profondita'")
for n in range(1, 11):
    reset()
    hanoi(n, "A", "C", "B")
    print("%3d | %6d | %7d | %8d | %6d | %11d | %d" % (n, moves, 2 ** n - 1, calls, sum(cnt), 3 * 2 ** n - 4, max_depth))

# La leggenda: 64 dischi, una mossa al secondo
print()
print("n = 64: 2^64 - 1 secondi sono piu' di %.1e anni" % ((2.0 ** 64 - 1) / (365.25 * 24 * 3600)))

1. sposta il disco 1 da A a C
2. sposta il disco 2 da A a B
3. sposta il disco 1 da C a B
4. sposta il disco 3 da A a C
5. sposta il disco 1 da B a A
6. sposta il disco 2 da B a C
7. sposta il disco 1 da A a C
7 mosse, 7 chiamate, profondita' della ricorsione = 3

chiamata                    | azione
Hanoi(3,A,C,B)              | chiama Hanoi(2,A,B,C)
  Hanoi(2,A,B,C)            | chiama Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | stampa: sposta il disco 1 da A a C, ritorna
  Hanoi(2,A,B,C)            | stampa: sposta il disco 2 da A a B, chiama Hanoi(1,C,B,A)
    Hanoi(1,C,B,A)          | stampa: sposta il disco 1 da C a B, ritorna
  Hanoi(2,A,B,C)            | ritorna
Hanoi(3,A,C,B)              | stampa: sposta il disco 3 da A a C, chiama Hanoi(2,B,C,A)
  Hanoi(2,B,C,A)            | chiama Hanoi(1,B,A,C)
    Hanoi(1,B,A,C)          | stampa: sposta il disco 1 da B a A, ritorna
  Hanoi(2,B,C,A)            | stampa: sposta il disco 2 da B a C, chiama Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)

**Prova tu.** Cambia il numero di dischi e conta le mosse.

In [19]:
n = 4        # <-- cambia il numero di dischi
reset()
show_moves = True
hanoi(n, "A", "C", "B")
show_moves = False
print("%d mosse = 2^%d - 1 = %d" % (moves, n, 2 ** n - 1))

1. sposta il disco 1 da A a B
2. sposta il disco 2 da A a C
3. sposta il disco 1 da B a C
4. sposta il disco 3 da A a B
5. sposta il disco 1 da C a A
6. sposta il disco 2 da C a B
7. sposta il disco 1 da A a B
8. sposta il disco 4 da A a C
9. sposta il disco 1 da B a C
10. sposta il disco 2 da B a A
11. sposta il disco 1 da C a A
12. sposta il disco 3 da B a C
13. sposta il disco 1 da A a B
14. sposta il disco 2 da A a C
15. sposta il disco 1 da B a C
15 mosse = 2^4 - 1 = 15
